In [42]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.width", 100)
sns.set_theme(style="whitegrid")

FINDINGS = Path("findings.md")
channels = Path("../data/channels.parquet")
clips = Path("../data/clips.parquet")
games = Path("../data/games.parquet")

chdf = pd.read_parquet(channels)
cdf = pd.read_parquet(clips)
gdf = pd.read_parquet(games)

print("channels : ", chdf.shape[0], "rows,", chdf.shape[1], "columns")
print("clips : ", cdf.shape[0], "rows,", cdf.shape[1], "columns")
print("games : ", gdf.shape[0], "rows,", gdf.shape[1], "columns")

channels :  877 rows, 11 columns
clips :  1992 rows, 17 columns
games :  20 rows, 4 columns


### Transforming the data

In [43]:
# TODO: remove personal data in data acquisition notebook
# remove uncessary columns
# id in chdf matches user_id in cdf, so we keep it.
chdf = chdf.drop(columns=["broadcaster_login", "broadcaster_name"])
cdf = cdf.drop(columns=["creator_id", "broadcaster_name", "creator_name", "thumbnail_url", "url", "embed_url"])
gdf = gdf.drop(columns=["box_art_url"])


# now we transform variables to proper types
chdf['broadcaster_id'] = pd.to_numeric(chdf["broadcaster_id"], errors="coerce").astype("Int64")
chdf['game_id'] = pd.to_numeric(chdf["game_id"], errors="coerce").astype("Int64")
chdf['game_name'] = chdf['game_name'].astype("string")
chdf['broadcaster_language'] = chdf['broadcaster_language'].astype("string")
chdf['title'] = chdf['title'].astype("string")
# Here we parse the tags column, which is a comma-separated string of tags, into a list of strings. We also strip whitespace from each tag.
chdf["tags"] = chdf["tags"].astype("string")
chdf['tags'] = chdf['tags'].str.split(",").apply(lambda x: [item.strip() for item in x] if isinstance(x, list) else x)
chdf['content_classification_labels'] = chdf['content_classification_labels'].astype("string")


# Clip IDs are strings
cdf["id"] = cdf["id"].astype("string")
cdf["broadcaster_id"] = pd.to_numeric(cdf["broadcaster_id"], errors="coerce").astype("Int64")
cdf["video_id"] = pd.to_numeric(cdf["video_id"], errors="coerce").astype("Int64")
cdf["game_id"] = pd.to_numeric(cdf["game_id"], errors="coerce").astype("Int64")
cdf["language"] = cdf["language"].astype("string")
cdf["title"] = cdf["title"].astype("string")
# Quantitative / Numerical features
cdf["view_count"] = pd.to_numeric(cdf["view_count"], errors="coerce").astype("Int64")
cdf["duration"] = pd.to_numeric(cdf["duration"], errors="coerce").astype("Float64")
# vod_offset is in seconds; Int64 accommodates clips where VOD no longer exists (null)
cdf["vod_offset"] = pd.to_numeric(cdf["vod_offset"], errors="coerce").astype("Int64")
cdf["created_at"] = pd.to_datetime(cdf["created_at"], utc=True)
cdf["is_featured"] = cdf["is_featured"].astype("boolean")

gdf["id"] = pd.to_numeric(gdf["id"], errors="coerce").astype("Int64")
gdf["name"] = gdf["name"].astype("string")
gdf["igdb_id"] = pd.to_numeric(gdf["igdb_id"], errors="coerce").astype("Int64")


print("channels dtypes:")
print(chdf.dtypes.to_string())
print("\nclips dtypes:")
print(cdf.dtypes.to_string())
print("\ngames dtypes:")
print(gdf.dtypes.to_string())

channels dtypes:
broadcaster_id                            Int64
broadcaster_language             string[python]
game_id                                   Int64
game_name                        string[python]
title                            string[python]
delay                                     int64
tags                                     object
content_classification_labels    string[python]
is_branded_content                         bool

clips dtypes:
id                     string[python]
broadcaster_id                  Int64
video_id                        Int64
game_id                         Int64
language               string[python]
title                  string[python]
view_count                      Int64
created_at        datetime64[ns, UTC]
duration                      Float64
vod_offset                      Int64
is_featured                   boolean

games dtypes:
id                  Int64
name       string[python]
igdb_id             Int64


In [46]:
# estimators of quantitative vars
print("delay descriptors : ", chdf['delay'].describe().round(2).to_string())
print("view_count descriptors : ", cdf['view_count'].describe().round(2).to_string())
print("duration descriptors : ", cdf['duration'].describe().round(2).to_string())
print("vod_offset descriptors : ", cdf['vod_offset'].describe().round(2).to_string())



delay descriptors :  count    877.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
view_count descriptors :  count      1992.0
mean      8266.01
std      14121.44
min          80.0
25%        936.75
50%        3672.5
75%       9748.25
max      215980.0
duration descriptors :  count    1992.0
mean      26.52
std       17.16
min         4.9
25%        11.5
50%        24.0
75%        34.6
max        60.0
vod_offset descriptors :  count      1515.0
mean     13728.24
std      17368.14
min          93.0
25%        4151.0
50%        9355.0
75%       17111.5
max      167974.0


we see that delay is 0 for all. This isn't very interesting, we will remove it from the df:

In [48]:
chdf = chdf.drop(columns =['delay'])

### Identifying collumns with missing values : 

In [49]:
for(df, name) in [(chdf, "channels"), (cdf, "clips"), (gdf, "games")]:
    print(f"\n{name} missing values:")
    missing = df.isna().sum()
    missing = missing[missing > 0]
    percentage_missing = (missing / df.shape[0]) * 100
    print(missing.to_string(),"\n\nPercentage of missing values:\n",  percentage_missing.round(2).to_string())



channels missing values:
game_id    5 

Percentage of missing values:
 game_id    0.57

clips missing values:
video_id      477
vod_offset    477 

Percentage of missing values:
 video_id      23.95
vod_offset    23.95

games missing values:
igdb_id    6 

Percentage of missing values:
 igdb_id    30.0


we see that igdb_id for game df, as well as video_id and vod_offset for clips df, are missing at high percentage. Furthermore, video_id and vod_offset are missing at the same rate, leading us to wonder if this is linked.

we should consider investigating the link :

In [50]:
both_missing = cdf["video_id"].isna() & cdf["vod_offset"].isna()
one_missing = cdf["video_id"].isna() ^ cdf["vod_offset"].isna()

print(f"Rows where both are missing: {both_missing.sum()}")
print(f"Rows where only one is missing: {one_missing.sum()}")

Rows where both are missing: 477
Rows where only one is missing: 0


this confirms our intuition. After some research, we learn that in Twitch's architecture, if a broadcaster does not archive past broadcasts (VODs) or if the 14–60 day VOD retention window expires, Twitch drops both the source video_id and the vod_offset simultaneously, while keeping the clip itself.

we also found that the missing igdb_id is for linking to Internet Game Database, to get game metadata. This is outside of our scope.

In any case these columns are not interesting to use. We can delete them :

In [52]:
cdf = cdf.drop(columns=["video_id", "vod_offset"])
gdf = gdf.drop(columns=["igdb_id"])

## Looking for impossible, extreme, ambiguous values

In [ ]:
impossible_clips = (
    (cdf["duration"] <= 0)                      # Clip has zero or negative duration
    | (cdf["duration"] > 60)                   #  Twitch clip minimum length is 5s, max is 60
    | (cdf["view_count"] < 0)                   # no egative views
    | (cdf["created_at"] > pd.Timestamp.now(tz="UTC"))  # Future timestamps or timestaps that are too old
    | (cdf["created_at"] < pd.Timestamp("2015-01-01", tz="UTC"))  # Timestamps that are too old
)

ambiguous_clips = (
    (cdf["view_count"] == 0)                    # Zero views (dead clip or tracking delay)
    | (cdf["duration"] < 5.0)                   #  Twitch clip minimum length is 5s, max is 60
    | (cdf["view_count"] > 1000000)           # Extreme outlier
)

print(f"Impossible rows  : {impossible_clips.sum():,} ({impossible_clips.mean():.2%})")
print(f"Ambiguous rows   : {ambiguous_clips.sum():,} ({ambiguous_clips.mean():.2%})")
print(f"Clean rows       : {(~(impossible_clips | ambiguous_clips)).sum():,} ({(~(impossible_clips | ambiguous_clips)).mean():.2%})\n")

if ambiguous_clips.any():
    print("Sample ambiguous clips:")
    print(cdf.loc[ambiguous_clips, ["id", "title", "duration", "view_count", "duration"]].head())

Impossible rows  : 0 (0.00%)
Ambiguous rows   : 50 (2.51%)
Clean rows       : 1,942 (97.49%)

Sample ambiguous clips:
                                                   id               title  duration  view_count  \
4         BusyTenaciousAdminDansGame-OR-2gRT9xxUTT9yC    CLAIM THAT WOMAN       4.9      101022   
11      BlightedGlamorousCookieKeepo-O1Lh3ruvSe00f1qN               зачем       4.9       69135   
18  AbnegateWimpyDragonfruitKappaPride-MyVBtaIxXm8...                オカズ🦶       4.9       56772   
19  CoweringAggressiveHerbsGrammarKing-BQTBTPeLT3a...       he just nuted       4.9       54744   
33  AuspiciousHelplessSangMrDestructoid-8jgD6lhw23...  i am not good girl       4.9       39859   

    duration  
4        4.9  
11       4.9  
18       4.9  
19       4.9  
33       4.9  


we see that the probelm is a duration of 4.9. This seems ok, because min duration is 5 so it's a small error we don't care about.

## Plotting Data

## Save Findings.md
